# ALPHA构建
对于上述每一个组合，控制市值等因子，计算其ALPHA的显著性        

## 导入库

In [29]:
import warnings
from pathlib import Path
import polars as pl
import numpy as np
import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS
import plotly.express as px

## 超参数

In [30]:
TASK_ID_PREFIX = 'lstm_short'  # 任务id前缀
SAVE_BASE_DIR = f'/home/frank/files/programs/GraduationThesis/empirical/{TASK_ID_PREFIX}' # 保存基本路径
SAVE_BASELINE_REG_DIR = SAVE_BASE_DIR + '/baseline_reg'
SAVE = True # 是否保存数据

RISK_FREE_RATE = 0.015 / 12 # 无风险利率  

## 读取数据(测试)
### 读取FF5数据  

FF5数据是一个月度时序数据，统计了每个月，FF5因子按照2*3方式构建组合的对冲组合的收益。使用该数据来计算alpha  

从json导入（测试）  

In [31]:
# 路径
ff5_files = Path(SAVE_BASE_DIR).parent.glob(f'STK_MKT_FIVEFACMONTH_*.json')
ff5_files = list(ff5_files)

# 读取数据  
ff5 = [pl.scan_ndjson(file) for file in ff5_files]
ff5 = pl.concat(ff5, how='vertical')
ff5.head().collect()


MarkettypeID,TradingMonth,Portfolios,RiskPremium1,SMB1,HML1,RMW1,CMA1
str,str,i64,f64,f64,f64,f64,f64
"""P9709""","""1997-01""",3,0.071179,0.033837,0.020607,-0.00408,-0.004712
"""P9709""","""1997-01""",2,0.071179,0.037394,0.012023,0.002056,-0.003199
"""P9709""","""1997-01""",1,0.071179,0.036629,0.009923,0.002077,-0.012824
"""P9714""","""1997-01""",1,0.071179,0.036629,0.009923,0.002077,-0.012824
"""P9714""","""1997-01""",3,0.071179,0.033837,0.020607,-0.00408,-0.004712


### 读取分桶数据  
- 分桶数据用来和FF5数据合并，计算alpha     
- 分桶表现数据用于合并alpha表现  



In [32]:
combined_series = pl.scan_parquet(SAVE_BASELINE_REG_DIR + '/基准回归-分桶市值加权收益.parquet')
performance = pl.read_parquet(SAVE_BASELINE_REG_DIR + '/基准回归-分桶表现.parquet')

## 处理数据：  

- 1.选用Portfolio == 1 的组合 (2*3构建)  
- 2.MarkettypeID == P9725：沪深A股和创业板和科创板 (不选京，否则没有早期年份)     
- 3.去除上述两列  
- 4.将TradingMonth列重命名为`date`    
- 5.因子重命名为`market_ret, smb, hml, rmw, cma`  

In [33]:
ff5 = ff5.filter((pl.col('Portfolios') == 1) & (pl.col('MarkettypeID') == 'P9714')).select(['TradingMonth', 'RiskPremium1', 'SMB1', 'HML1', 'RMW1', 'CMA1'])
ff5 = ff5.rename({'TradingMonth':'date', 'RiskPremium1':'market_ret', 'SMB1':'smb', 'HML1':'hml', 'RMW1':'rmw', 'CMA1':'cma'})
ff5 = ff5.with_columns(pl.col('date').str.strptime(pl.Date, '%Y-%m', strict=False))

In [34]:
ff5.head().collect()

date,market_ret,smb,hml,rmw,cma
date,f64,f64,f64,f64,f64
1997-01-01,0.071179,0.036629,0.009923,0.002077,-0.012824
1997-02-01,0.0509,0.042419,0.008079,-0.057175,0.024586
1997-03-01,0.184236,0.043714,-0.026034,0.037267,-0.096853
1997-04-01,0.08511,-0.047831,-0.028225,0.000637,-0.166661
1997-05-01,-0.081863,-0.038867,0.029129,0.041593,-0.017627


## 计算alpha  

### CAPM-ALPHA    
从FF5中获取market_ret列，形成时序数据`date-market_ret`  
将数据和combined_series合并，形成`date-code-bucket_id-ret-market_ret`表  

计算`(ret-risk_free_rate)~market_ret`回归的alpha（NW标准误）   

In [35]:
# 获取date-market_ret
market_ret = ff5.select('date','market_ret')

# 合并
joined_series = combined_series.join(market_ret, on='date', how='left')

## 按 bucket_id 分组回归（Polars map_groups）
coll = joined_series.collect()

def regress_one(s: pl.DataFrame) -> pl.DataFrame:
    g = s.to_pandas()
    bid = g['bucket_id'].iloc[0]  # 安全取分组 id，避免 Polars 内索引触发 panic
    try:
        y = g['weighted_sum_ret'].to_numpy() - RISK_FREE_RATE
        x = g['market_ret'].to_numpy()
        x_with_constant = sm.add_constant(x)
        results = sm.OLS(y, x_with_constant).fit(cov_type='HAC', cov_kwds={'maxlags': 4})
        return pl.DataFrame({
            'bucket_id': [bid],
            'camp-alpha': [float(results.params['const'])],
            't': [float(results.tvalues['const'])],
            'p': [float(results.pvalues['const'])],
        })
    except Exception as e:
        warnings.warn(f"计算{bid}时发生错误: {e}")
        return pl.DataFrame({
            'bucket_id': [bid],
            'camp-alpha': [0.0],
            't': [0.0],
            'p': [1.0],
        })

alpha_table = coll.group_by('bucket_id').map_groups(regress_one)

/tmp/ipykernel_1956/3963323801.py:25: UserWarning: 计算3时发生错误: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices
  warnings.warn(f"计算{bid}时发生错误: {e}")
/tmp/ipykernel_1956/3963323801.py:25: UserWarning: 计算对冲组合时发生错误: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices
  warnings.warn(f"计算{bid}时发生错误: {e}")
/tmp/ipykernel_1956/3963323801.py:25: UserWarning: 计算0时发生错误: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices
  warnings.warn(f"计算{bid}时发生错误: {e}")


格式化输出：   
[ ]内为HAC-t，()内为p值    

In [36]:
# 格式化输出（4 位有效数字）
alpha_table = alpha_table.with_columns(
    pl.col('camp-alpha').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('camp-alpha'),
    pl.col('t').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('t'),
    pl.col('p').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('p'),
)

# 为t和p添加括号  
alpha_table = alpha_table.select(
    pl.col('bucket_id'),
    pl.col('camp-alpha'),
    (pl.lit('[') + pl.col('t') + pl.lit(']')).alias('t'),
    (pl.lit('(') + pl.col('p') + pl.lit(')')).alias('p'),
)

# 将alpha、t、p居中对齐
alpha_table = alpha_table.with_columns(
    pl.col('camp-alpha').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('camp-alpha'),
    pl.col('t').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('t'),
    pl.col('p').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('p'),
)

# 将alpha、t、p合并为一行
alpha_table = alpha_table.select(
    pl.col('bucket_id'),
    (pl.col('camp-alpha') + pl.lit('\n') + pl.col('t') + pl.lit('\n') + pl.col('p')).alias('camp-alpha'),
)

alpha_table.sort('bucket_id')

alpha_table

bucket_id,camp-alpha
str,str
"""3""",""" 0 [0] …"
"""对冲组合""",""" 0 [0] …"
"""0""",""" 0 [0] …"


### FF3-ALPHA  
使用FAMA-FRENCH模型计算ALPHA  

与CAPM类似，使用`market_ret, smb, hml, rmw, cma`作为自变量，计算`weighted_sum_ret`的alpha    

In [37]:
# 获取ff3数据
ff3 = ff5.select('date', 'market_ret', 'smb', 'hml')

#连接
joined_series = combined_series.join(ff3, on='date', how='left')

# 按bucket_id分组回归
coll = joined_series.collect()

def regress_ff3(s: pl.DataFrame) -> pl.DataFrame:
    g = s.to_pandas()
    bid = g['bucket_id'].iloc[0]
    try:
        y = g['weighted_sum_ret'].to_numpy() - RISK_FREE_RATE
        x = g[['market_ret', 'smb', 'hml']].to_numpy()
        x_with_constant = sm.add_constant(x)
        results = sm.OLS(y, x_with_constant).fit(cov_type='HAC', cov_kwds={'maxlags': 4})
        return pl.DataFrame({
            'bucket_id': [bid],
            'ff3-alpha': [float(results.params['const'])],
            't': [float(results.tvalues['const'])],
            'p': [float(results.pvalues['const'])],
        })
    except Exception as e:
        warnings.warn(f"计算{bid}时发生错误: {e}")
        return pl.DataFrame({
            'bucket_id': [bid],
            'ff3-alpha': [0.0],
            't': [0.0],
            'p': [1.0],
        })

ff3_alpha_table = coll.group_by('bucket_id').map_groups(regress_ff3)

/tmp/ipykernel_1956/2997979645.py:25: UserWarning: 计算对冲组合时发生错误: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices
  warnings.warn(f"计算{bid}时发生错误: {e}")
/tmp/ipykernel_1956/2997979645.py:25: UserWarning: 计算3时发生错误: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices
  warnings.warn(f"计算{bid}时发生错误: {e}")
/tmp/ipykernel_1956/2997979645.py:25: UserWarning: 计算0时发生错误: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices
  warnings.warn(f"计算{bid}时发生错误: {e}")


格式化输出

In [38]:
# FF3-alpha 格式化输出（4 位有效数字、括号、居中对齐、合并一行）
ff3_alpha_table = ff3_alpha_table.with_columns(
    pl.col('ff3-alpha').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('ff3-alpha'),
    pl.col('t').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('t'),
    pl.col('p').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('p'),
)
ff3_alpha_table = ff3_alpha_table.select(
    pl.col('bucket_id'),
    pl.col('ff3-alpha'),
    (pl.lit('[') + pl.col('t') + pl.lit(']')).alias('t'),
    (pl.lit('(') + pl.col('p') + pl.lit(')')).alias('p'),
)
ff3_alpha_table = ff3_alpha_table.with_columns(
    pl.col('ff3-alpha').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('ff3-alpha'),
    pl.col('t').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('t'),
    pl.col('p').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('p'),
)
ff3_alpha_table = ff3_alpha_table.select(
    pl.col('bucket_id'),
    (pl.col('ff3-alpha') + pl.lit('\n') + pl.col('t') + pl.lit('\n') + pl.col('p')).alias('ff3-alpha'),
)

ff3_alpha_table.sort('bucket_id')

bucket_id,ff3-alpha
str,str
"""0""",""" 0 [0] …"
"""3""",""" 0 [0] …"
"""对冲组合""",""" 0 [0] …"


### FF5-ALPHA 

In [39]:
# 获取 FF5 数据（五因子）
ff5_factors = ff5.select('date', 'market_ret', 'smb', 'hml', 'rmw', 'cma')

# 连接
joined_series = combined_series.join(ff5_factors, on='date', how='left')

# 按 bucket_id 分组回归
coll = joined_series.collect()

def regress_ff5(s: pl.DataFrame) -> pl.DataFrame:
    g = s.to_pandas()
    bid = g['bucket_id'].iloc[0]
    try:
        y = g['weighted_sum_ret'].to_numpy() - RISK_FREE_RATE
        x = g[['market_ret', 'smb', 'hml', 'rmw', 'cma']].to_numpy()
        x_with_constant = sm.add_constant(x)
        results = sm.OLS(y, x_with_constant).fit(cov_type='HAC', cov_kwds={'maxlags': 4})
        return pl.DataFrame({
            'bucket_id': [bid],
            'ff5-alpha': [float(results.params['const'])],
            't': [float(results.tvalues['const'])],
            'p': [float(results.pvalues['const'])],
        })
    except Exception as e:
        warnings.warn(f"计算{bid}时发生错误: {e}")
        return pl.DataFrame({
            'bucket_id': [bid],
            'ff5-alpha': [0.0],
            't': [0.0],
            'p': [1.0],
        })

ff5_alpha_table = coll.group_by('bucket_id').map_groups(regress_ff5)
ff5_alpha_table

/tmp/ipykernel_1956/930366058.py:25: UserWarning: 计算0时发生错误: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices
  warnings.warn(f"计算{bid}时发生错误: {e}")
/tmp/ipykernel_1956/930366058.py:25: UserWarning: 计算对冲组合时发生错误: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices
  warnings.warn(f"计算{bid}时发生错误: {e}")
/tmp/ipykernel_1956/930366058.py:25: UserWarning: 计算3时发生错误: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices
  warnings.warn(f"计算{bid}时发生错误: {e}")


bucket_id,ff5-alpha,t,p
str,f64,f64,f64
"""0""",0.0,0.0,1.0
"""对冲组合""",0.0,0.0,1.0
"""3""",0.0,0.0,1.0


格式化输出

In [40]:
# FF5-alpha 格式化输出（4 位有效数字、括号、居中对齐、合并一行）
ff5_alpha_table = ff5_alpha_table.with_columns(
    pl.col('ff5-alpha').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('ff5-alpha'),
    pl.col('t').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('t'),
    pl.col('p').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('p'),
)
ff5_alpha_table = ff5_alpha_table.select(
    pl.col('bucket_id'),
    pl.col('ff5-alpha'),
    (pl.lit('[') + pl.col('t') + pl.lit(']')).alias('t'),
    (pl.lit('(') + pl.col('p') + pl.lit(')')).alias('p'),
)
ff5_alpha_table = ff5_alpha_table.with_columns(
    pl.col('ff5-alpha').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('ff5-alpha'),
    pl.col('t').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('t'),
    pl.col('p').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('p'),
)
ff5_alpha_table = ff5_alpha_table.select(
    pl.col('bucket_id'),
    (pl.col('ff5-alpha') + pl.lit('\n') + pl.col('t') + pl.lit('\n') + pl.col('p')).alias('ff5-alpha'),
)

ff5_alpha_table.sort('bucket_id')

bucket_id,ff5-alpha
str,str
"""0""",""" 0 [0] …"
"""3""",""" 0 [0] …"
"""对冲组合""",""" 0 [0] …"


## 合并所有表现

In [41]:
performance = performance.join(alpha_table, on='bucket_id', how='left')
performance = performance.join(ff3_alpha_table, on='bucket_id', how='left')
performance = performance.join(ff5_alpha_table, on='bucket_id', how='left')
performance = performance.sort('bucket_id')
performance.head()


bucket_id,mean_return,sharp,camp-alpha,ff3-alpha,ff5-alpha
str,str,str,str,str,str
"""0""",""" -0.002002 [-1.587] (0…","""-0.2074""",""" 0 [0] …",""" 0 [0] …",""" 0 [0] …"
"""3""",""" 0.004728 [1.314] (0…","""0.07195""",""" 0 [0] …",""" 0 [0] …",""" 0 [0] …"
"""对冲组合""",""" 0.002002 [1.587] (0…","""0.04797""",""" 0 [0] …",""" 0 [0] …",""" 0 [0] …"


In [42]:
if SAVE:
    performance.write_parquet(SAVE_BASELINE_REG_DIR + '/基准回归-分桶表现.parquet')